In [13]:
import pandas as pd
from langchain_ollama import OllamaLLM
import re
import ast

stigmatizingWords = [['challenging', 'uncooperative', 'noncompliant', 'nonadherent', 'refused', 'frequent visitor to ED', 'narcotic dependence', 'obese', 'alcoholic', 'inconsistent responses']]

model = OllamaLLM(model="llama3.2")

def askOllama(prompt):
    result = model.invoke(input=prompt)
    return result

def cleanOllamaOutput(output):
    pattern = r"\[.*?\]"
    
    matches = re.findall(pattern, output, re.DOTALL)
    a = matches[0].replace("\n", "")
    escaped_string = re.sub(r"(?<=\w)'(?=\w)", r"\'", a)
    result = re.sub(r"\([^()]*\)", "", escaped_string)
    return ast.literal_eval(result.replace("\\n", "").replace("\\\\", "\\"))

df = pd.read_csv("/Users/sagewong/git/StigmatizingLanguageProject/Application/FinalAnnotatedData.csv")
clinicalNote = df.iloc[0]['Completion']

allList = []
for index, i in enumerate(stigmatizingWords):
    clinicalNote = df.iloc[index]["Completion"]
    clinicalNote = re.sub(r'^.*?\*\*History of Present Illness:\*\*', '', clinicalNote, flags=re.DOTALL)
    sentences = clinicalNote.split("**")
    sentences = [item for part in sentences for item in part.split("-")]
    for word in i:
        if len([j for j in sentences if word in j]) > 0:
            text = [j for j in sentences if word in j][0]
            allList.append([word, text])
def group_by_second_index(data):
    result = {}

    for element in data:
        key = element[1]  # The second index (the grouping key)
        value = element[0]  # The first index (the value for the key)

        if key in result:
            result[key].append(value)  # If the key exists, append the value to the list
        else:
            result[key] = [value]  # If the key doesn't exist, create a new list with the value

    return result
grouped_list = group_by_second_index(allList)

In [17]:
newDict = {}
for key, value in grouped_list.items():
    replacingPrompt = "You are a professional linguist whose job is to replace stigmatizing language in clinical notes. If you see labels such as diabetic or abuser, replace these labels with person first language such as \"person who has diabetes\" or\"person with a substance abuse disorder\". If you see words like challenging or uncooperative, replace them with more respectful alternatives. Here is the sentence: " + str(key) + " And here is are the stigmatizing words you must replace: " + str(value) + ". Return to me a JSON object containing only the corrected sentence in a list"
    while True:
        try:
            newDict[key] = cleanOllamaOutput(askOllama(replacingPrompt))[0]
            break
        except:
            pass
newDict

{' Mohammad, a challenging patient with a history of uncooperative behavior, presents with concerns about his significantly short stature compared to peers. Despite recommendations for nutritional interventions, Mohammad has been noncompliant, citing his preferences for junk food. His mother reports frequent cursing at nurses during previous medical consultations, further complicating collaborative treatment efforts.\n\n': 'Mohammad, a person with diabetes presents with concerns about his significantly short stature compared to peers. Despite recommendations for nutritional interventions, Mohammad has been resistant to change, citing his preferences for junk food. His mother reports frequent cursing at nurses during previous medical consultations, further complicating collaborative treatment efforts.',
 ' Throughout his stay, Mohammad was a difficult patient, reflecting a pattern of challenging interactions and unreliable historical recounting. Compliance issues were evident, as he ref

In [23]:
newClinicalNote = clinicalNote
for key, value in newDict.items():
    newClinicalNote = newClinicalNote.replace(key, value)
print(list(newDict.values()))

['Mohammad, a person with diabetes presents with concerns about his significantly short stature compared to peers. Despite recommendations for nutritional interventions, Mohammad has been resistant to change, citing his preferences for junk food. His mother reports frequent cursing at nurses during previous medical consultations, further complicating collaborative treatment efforts.', 'Throughout his stay, Mohammad was a difficult patient, reflecting a pattern of challenging interactions and unreliable historical recounting. Compliance issues were evident, as he declined several exams and was resistant to modifying his diet, despite clear evidence linking his nutritional habits to his short stature. Cursing at staff persisted, complicating our attempts to provide care.']
